# 07 · Rollout animations (GIFs for the project page)

Builds side-by-side animated GIFs of autoregressive rollouts — ground truth next to the models you pick —
and writes them to `artifacts/gifs/<pde>_rollout.gif`. Copy the ones you want on the project page to `docs/static/gifs/`.

Uses the same inputs as `03_qualitative_rollouts.ipynb`: `data/datasets/<pde>` (only for the field names) and
`experiments/<pde>/results/<MODEL>_<pde>_traj.pkl` + `ground_truth_<pde>_traj.pkl` (one-sample rollouts).
The GIF writer (`wavelet_flow_matching/utils/animation.py`) needs only numpy, matplotlib and Pillow;
titles are typeset in LaTeX style (Computer Modern), so labels can use math, e.g. `r"FM$_{pixel}$"`.

Set `well_dataset_name` below and run all; repeat for each system.

In [ ]:
import os
import sys
import pickle
from pathlib import Path

import numpy as np
from IPython.display import display

PROJECT_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
sys.path.insert(0, str(PROJECT_ROOT))

from wavelet_flow_matching.utils.animation import preview, rollout_gif
from wavelet_flow_matching.utils.rollout import get_gt_frame, get_pred_frame

#well_dataset_name = "turbulent_radiative_layer_2D"
well_dataset_name = "rayleigh_benard"
#well_dataset_name = "active_matter"

RESULT_DIR = PROJECT_ROOT / "experiments" / well_dataset_name / "results"
OUT_DIR    = PROJECT_ROOT / "artifacts" / "gifs"
OUT_DIR.mkdir(parents=True, exist_ok=True)

## Settings

* **Panels** — keys are the paper model names of the rollout files, values the titles drawn above each panel
  (LaTeX math allowed). Ground truth is always the first panel and sets the shared colour range.
* **Size on screen** — `panel_width` / `panel_height` (pixels per panel) and `fontsize` (points) in `DISPLAY`.
  Setting both stretches the field to that box, like `aspect="auto"` in the paper figures; `panel_height=None`
  keeps the field's own aspect ratio.
* **File size** — grows with panels × pixels × frames. Aim for < 25 MB per GIF: raise `stride`, set
  `MAX_FRAMES`, lower `panel_width` or drop a panel (the size is printed at the end).

In [ ]:
# Mother wavelet of the WFM panels, per system (same choice as notebook 03).
WAVELET_PER_PDE = {
    "turbulent_radiative_layer_2D": {"1scale": "db2",  "3scale": "db2"},
    "rayleigh_benard":              {"1scale": "db2",  "3scale": "db2"},
    "active_matter":                {"1scale": "haar", "3scale": "db2"},
}
SEL = WAVELET_PER_PDE[well_dataset_name]

# rollout file name (paper model name) -> panel title
PANELS = {
    "FM_pixel"                    : r"FM$_{pixel}$",
    f"WFM_1scale_{SEL['1scale']}" : r"WFM 1-scale (ours)",
    f"WFM_3scale_{SEL['3scale']}" : r"WFM 3-scale (ours)",
    "FM_fourier"                  : r"FM$_{fourier}$",
    #"LatentFM"                   : r"LatentFM$_{\div 64}$",   # RB only
    #"WDNO"                       : r"WDNO",
    #"FNO"                        : r"FNO",
}

# Per-system display options (colour map and orientation match the paper figures)
DISPLAY = {
    "turbulent_radiative_layer_2D": dict(cmap="RdBu_r",  transpose=False, panel_width=420, panel_height=210, stride=1),
    "rayleigh_benard":              dict(cmap="RdBu_r",  transpose=True,  panel_width=480, panel_height=240, stride=2),
    "active_matter":                dict(cmap="viridis", transpose=False, panel_width=320, panel_height=None, stride=1),
}[well_dataset_name]
DISPLAY.update(fontsize=16, usetex=False)   # usetex=True uses your LaTeX install (slower)

TRAJ_IDX      = 2      # test trajectory (notebook 03 uses 2)
ENS_IDX       = 0      # ensemble member
FIELD_IDX     = 0      # physical field to animate
N_STEPS_INPUT = 4      # context frames used by the model
MAX_FRAMES    = None   # cap the number of frames (None = whole rollout)
FPS           = 10
HOLD_LAST     = 10     # repeat the last frame so the loop pauses before restarting

In [ ]:
def load(path):
    with open(path, "rb") as f:
        data = pickle.load(f)
    return data["preds"] if isinstance(data, dict) else data

truth_traj = load(RESULT_DIR / f"ground_truth_{well_dataset_name}_traj.pkl")

rollouts = {}
for name in PANELS:
    path = RESULT_DIR / f"{name}_{well_dataset_name}_traj.pkl"
    if path.exists():
        rollouts[name] = load(path)
        print("loaded", path.name)
    else:
        print("missing", path.name, "- skipped")

# Field name for the file name / title (falls back to the index if the dataset is not available)
try:
    from wavelet_flow_matching.utils.well_data import get_well_dataset
    ds = get_well_dataset(path=str(PROJECT_ROOT / "data" / "datasets"), physics=well_dataset_name,
                          split="test", steps=-1, transform=["log_scalars"], augment=[])
    field_names = [f for group in ds.metadata.field_names.values() for f in group]
    FIELD_NAME = field_names[FIELD_IDX]
except Exception as e:
    print("dataset not available, using the field index:", type(e).__name__)
    FIELD_NAME = f"field{FIELD_IDX}"
print("field:", FIELD_NAME)

In [ ]:
n_rollout = truth_traj.shape[1] - N_STEPS_INPUT
leads = range(1, n_rollout + 1)

def as_np(x):
    return x.detach().cpu().numpy() if hasattr(x, "detach") else np.asarray(x)

fields = {"Ground truth": np.stack([as_np(get_gt_frame(truth_traj, TRAJ_IDX, t, N_STEPS_INPUT, FIELD_IDX)) for t in leads])}
for name, label in PANELS.items():
    if name in rollouts:
        fields[label] = np.stack([as_np(get_pred_frame(rollouts[name], TRAJ_IDX, ENS_IDX, t, FIELD_IDX)) for t in leads])

out = OUT_DIR / f"{well_dataset_name}_rollout.gif"
rollout_gif(fields, out, fps=FPS, hold_last=HOLD_LAST, max_frames=MAX_FRAMES, **DISPLAY)

size_mb = out.stat().st_size / 1e6
print(f"wrote {out.relative_to(PROJECT_ROOT)}  ({size_mb:.1f} MB, field '{FIELD_NAME}')")
if size_mb > 25:
    print("large GIF: raise `stride`, set MAX_FRAMES or lower `panel_width` for the web")

# First / middle / last frame only, so the notebook does not embed the whole GIF.
# Open the .gif file to see the animation.
display(preview(out))

## Other variants

* **Another field / trajectory** — change `FIELD_IDX` or `TRAJ_IDX` and rerun the last cell. To keep several,
  rename `out`, e.g. `OUT_DIR / f"{well_dataset_name}_{FIELD_NAME}_rollout.gif"`.
* **Project page** — copy `artifacts/gifs/<pde>_rollout.gif` to `docs/static/gifs/` (same name).
* **MP4 instead of GIF** — `wavelet_flow_matching.utils.plot.draw_movie` writes `.mp4` (needs `moviepy`).